In [1]:
import os
import sys
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline

sys.path.append(os.path.abspath("../src"))
from preprocessing import build_preprocessor

# Load dataset
df = pd.read_csv("../data/student_performance.csv")
X = df.drop(columns=["student_id", "passed"])
y = df["passed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = [
    "study_hours_per_week",
    "attendance_rate",
    "previous_gpa",
    "assignments_completed",
    "midterm_score",
]
categorical_features = ["extracurricular"]
preprocessor = build_preprocessor(numeric_features, categorical_features)

# Setup model and search grid (note the 'classifier__' prefix for pipeline steps)
model = RandomForestClassifier(random_state=42)
param_grid = {
    "classifier__n_estimators": [50, 100, 200],
    "classifier__max_depth": [None, 3, 5, 10],
    "classifier__min_samples_split": [2, 5],
}

pipe = Pipeline(
    steps=[("preprocessor", preprocessor), ("classifier", model)]
)

# Run Grid Search with 3-fold cross-validation
grid_search = GridSearchCV(pipe, param_grid, cv=3, scoring="f1", n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best Parameters Found:")
print(grid_search.best_params_)
print(f"Best CV F1 Score: {grid_search.best_score_:.4f}")

# Evaluate the best tuned model on the held-out test set
best_model = grid_search.best_estimator_
test_preds = best_model.predict(X_test)

print("\nTest Set Evaluation with Best Tuned Model:")
print(classification_report(y_test, test_preds))

Best Parameters Found:
{'classifier__max_depth': None, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 50}
Best CV F1 Score: 0.9333

Test Set Evaluation with Best Tuned Model:
              precision    recall  f1-score   support

           0       1.00      0.33      0.50         3
           1       0.60      1.00      0.75         3

    accuracy                           0.67         6
   macro avg       0.80      0.67      0.62         6
weighted avg       0.80      0.67      0.62         6

